## Projeto de Python para Finanças

In [78]:
# instalação
# pip install yfinance pandas numpy plotly nbformat

### Parte 1: Obter cotações e construção de carteira

In [79]:
import yfinance as yf
import pandas as pd
from datetime import datetime, timedelta

acoes = ["ITUB4.SA", "PETR4.SA", "VALE3.SA", "IVVB11.SA"]
# IBOVESPA -> ^BVSP
# SP500 -> ^GSPC
# Dólar -> BRL=X
# Ouro (dolarizado) -> GC=F
indices = ["^BVSP", "^GSPC", "BRL=X", "GC=F"]

data_inicio = datetime.now() - timedelta(days=730)
data_inicio = data_inicio.strftime("%Y-%m-%d")
data_fim = datetime.now().strftime("%Y-%m-%d")

def pegar_cotacoes(lista_tickers, data_inicio, data_fim):
    df = yf.download(lista_tickers, start=data_inicio, end=data_fim, auto_adjust=False)
    df = df["Adj Close"]
    df = df.ffill()
    df = df.dropna()
    return df

lista_tickers = acoes + indices
df_cotacoes = pegar_cotacoes(lista_tickers, data_inicio, data_fim)
display(df_cotacoes)

[*********************100%***********************]  8 of 8 completed


Ticker,BRL=X,GC=F,ITUB4.SA,IVVB11.SA,PETR4.SA,VALE3.SA,^BVSP,^GSPC
Date,,,,,,,,
2024-08-07,5.6555,2390.500000,25.231869,325.809998,28.448849,50.092228,127514.0,5199.500000
2024-08-08,5.6358,2422.199951,25.142212,330.720001,28.903784,49.785717,128661.0,5319.310059
2024-08-09,5.5461,2432.100098,25.822142,327.899994,28.637096,49.969624,130615.0,5344.160156
2024-08-12,5.5063,2462.399902,25.859491,327.450012,29.288116,49.715656,131116.0,5344.390137
2024-08-13,5.4919,2466.699951,26.591721,329.899994,29.107714,49.496723,132398.0,5434.430176
...,...,...,...,...,...,...,...,...
2026-08-03,5.0727,4033.699951,43.169998,435.829987,43.049999,74.639999,178000.0,7600.500000
2026-08-04,5.1029,4095.399902,42.099998,447.549988,42.500000,76.309998,177895.0,7736.520020
2026-08-05,5.1437,4245.799805,42.380001,446.510010,41.930000,76.660004,177726.0,7723.549805


In [80]:
# carteira em termos de valores financeiros
dic_carteira = {
    "ITUB4.SA": 5000,
    "VALE3.SA": 3000,
    "PETR4.SA": 4000,
    "IVVB11.SA": 6000
}

df_carteira = pd.DataFrame()
total_investido = sum(dic_carteira.values())

for ativo in dic_carteira:
    preco_inicial_ativo = df_cotacoes[ativo].iloc[0]
    qtde_acoes = dic_carteira[ativo] / preco_inicial_ativo
    df_carteira[ativo] = df_cotacoes[ativo] * qtde_acoes
    

df_carteira["Total"] = df_carteira.sum(axis=1)
display(df_carteira)


,ITUB4.SA,VALE3.SA,PETR4.SA,IVVB11.SA,Total
Date,,,,,
2024-08-07,5000.000000,3000.000000,4000.000000,6000.000000,18000.000000
2024-08-08,4982.233415,2981.643205,4063.965340,6090.420866,18118.262825
2024-08-09,5116.969716,2992.657282,4026.468232,6038.488623,18174.583852
2024-08-12,5124.371011,2977.447300,4118.003753,6030.201921,18250.023986
2024-08-13,5269.471091,2964.335500,4092.638543,6075.319905,18401.765038
...,...,...,...,...,...
2026-08-03,8554.657328,4470.154501,6052.968913,8026.088638,27103.869380
2026-08-04,8342.623945,4570.169907,5975.637244,8241.919974,27130.351071
2026-08-05,8398.109846,4591.131608,5895.493447,8222.768112,27107.503013


### Parte 2: Rentabilidade e Comparação com Benchmarks

In [81]:
df_cotacoes["SP500 (R$)"] = df_cotacoes["^GSPC"] * df_cotacoes["BRL=X"]
df_cotacoes["Ouro (R$)"] = df_cotacoes["GC=F"] * df_cotacoes["BRL=X"]
df_cotacoes["Dólar"] = df_cotacoes["BRL=X"]

df_cotacoes = df_cotacoes.drop(columns=["BRL=X", "GC=F", "^GSPC"])
display(df_cotacoes)

Ticker,ITUB4.SA,IVVB11.SA,PETR4.SA,VALE3.SA,^BVSP,SP500 (R$),Ouro (R$),Dólar
Date,,,,,,,,
2024-08-07,25.231869,325.809998,28.448849,50.092228,127514.0,29405.771913,13519.472595,5.6555
2024-08-08,25.142212,330.720001,28.903784,49.785717,128661.0,29978.567015,13651.034206,5.6358
2024-08-09,25.822142,327.899994,28.637096,49.969624,130615.0,29639.247389,13488.670691,5.5461
2024-08-12,25.859491,327.450012,29.288116,49.715656,131116.0,29427.815263,13558.712515,5.5063
2024-08-13,26.591721,329.899994,29.107714,49.496723,132398.0,29845.346904,13546.869381,5.4919
...,...,...,...,...,...,...,...,...
2026-08-03,43.169998,435.829987,43.049999,74.639999,178000.0,38555.056530,20461.749838,5.0727
2026-08-04,42.099998,447.549988,42.500000,76.309998,177895.0,39478.688226,20898.416277,5.1029
2026-08-05,42.380001,446.510010,41.930000,76.660004,177726.0,39727.624079,21839.120977,5.1437


In [82]:
# valor_final = 18102
# valor_inicial = 18000
# variacao_reais = valor_final - valor_inicial
# print(variacao_reais)

# variacao_percentual = valor_final / valor_inicial - 1
# print(variacao_percentual)

def calcular_retorno(df):
    retorno = df.iloc[-1] / df.iloc[0] - 1
    return retorno

display(calcular_retorno(df_carteira))
display(calcular_retorno(df_cotacoes))

ITUB4.SA     0.657824
VALE3.SA     0.505024
PETR4.SA     0.480904
IVVB11.SA    0.362604
Total        0.494635
dtype: float64

Ticker
ITUB4.SA      0.657824
IVVB11.SA     0.362604
PETR4.SA      0.480904
VALE3.SA      0.505024
^BVSP         0.376680
SP500 (R$)    0.347274
Ouro (R$)     0.612305
Dólar        -0.091415
dtype: float64

In [83]:
df_comparacao = df_cotacoes.drop(columns=acoes)
df_comparacao["Carteira"] = df_carteira["Total"]
display(df_comparacao)
print(calcular_retorno(df_comparacao))

Ticker,^BVSP,SP500 (R$),Ouro (R$),Dólar,Carteira
Date,,,,,
2024-08-07,127514.0,29405.771913,13519.472595,5.6555,18000.000000
2024-08-08,128661.0,29978.567015,13651.034206,5.6358,18118.262825
2024-08-09,130615.0,29639.247389,13488.670691,5.5461,18174.583852
2024-08-12,131116.0,29427.815263,13558.712515,5.5063,18250.023986
2024-08-13,132398.0,29845.346904,13546.869381,5.4919,18401.765038
...,...,...,...,...,...
2026-08-03,178000.0,38555.056530,20461.749838,5.0727,27103.869380
2026-08-04,177895.0,39478.688226,20898.416277,5.1029,27130.351071
2026-08-05,177726.0,39727.624079,21839.120977,5.1437,27107.503013


Ticker
^BVSP         0.376680
SP500 (R$)    0.347274
Ouro (R$)     0.612305
Dólar        -0.091415
Carteira      0.494635
dtype: float64


In [84]:
df_comparacao = (df_comparacao / df_comparacao.iloc[0] - 1) *100
display(df_comparacao)

import plotly.express as px

grafico = px.line(df_comparacao, x=df_comparacao.index, y=df_comparacao.columns)
grafico.update_layout(template="plotly_dark")
grafico.show()

Ticker,^BVSP,SP500 (R$),Ouro (R$),Dólar,Carteira
Date,,,,,
2024-08-07,0.000000,0.000000,0.000000,0.000000,0.000000
2024-08-08,0.899509,1.947900,0.973127,-0.348334,0.657016
2024-08-09,2.431890,0.793978,-0.227834,-1.934397,0.969910
2024-08-12,2.824788,0.074963,0.290247,-2.638139,1.389022
2024-08-13,3.830168,1.494860,0.202647,-2.892759,2.232028
...,...,...,...,...,...
2026-08-03,39.592515,31.113907,51.350208,-10.305011,50.577052
2026-08-04,39.510171,34.254895,54.580115,-9.771018,50.724173
2026-08-05,39.377637,35.101449,61.538261,-9.049595,50.597239


### Parte 3: Análise de Risco

In [94]:
import numpy as np

# correlação
df_cotacoes["Carteira"] = df_carteira["Total"]

# rentabilidade diária -> função logaritmica
tabela_rentabilidade_diaria = df_cotacoes / df_cotacoes.shift(1)
tabela_rentabilidade_diaria = np.log(tabela_rentabilidade_diaria).dropna()
tabela_correlacao = tabela_rentabilidade_diaria.corr()

grafico_correlacao = px.imshow(tabela_correlacao, text_auto=True, color_continuous_scale="greens")
grafico_correlacao.update_layout(template="plotly_dark")
grafico_correlacao.show()

# variância do retornos diários do ativo (aplicando a função logarítmica)
tabela_volatilidade = tabela_rentabilidade_diaria.std() * np.sqrt(252)

display(tabela_volatilidade)

Ticker
ITUB4.SA      0.217569
IVVB11.SA     0.154244
PETR4.SA      0.240562
VALE3.SA      0.251810
^BVSP         0.159452
SP500 (R$)    0.215505
Ouro (R$)     0.265726
Dólar         0.130059
Carteira      0.117290
dtype: float64

### Parte 4: Análise Técnica e Indicadores

In [102]:
ticker = "VALE3.SA"

import yfinance as yf
import plotly.graph_objects as go

df = yf.download(ticker, "2020-01-01", "2025-12-31", multi_level_index=False)
display(df)

# média móvel de 50 dias
df["MM50"] = df["Close"].rolling(50).mean()

# média móvel de 200 dias
df["MM200"] = df["Close"].rolling(200).mean()

grafico = go.Figure()

# adicionar as series do gráfico que você quer (os subgráficos)
grafico.add_trace(go.Candlestick(x=df.index, open=df["Open"], close=df["Close"], high=df["High"], low=df["Low"], name="Price"))

# adicionar a serie da MM50
grafico.add_trace(go.Scatter(x=df.index, y=df["MM50"], name= "MM50", line={"color": "blue", "width": 1}))

# adicionar a serie da MM200
grafico.add_trace(go.Scatter(x=df.index, y=df["MM200"], name= "MM200", line={"color": "yellow", "width": 1}))

grafico.update_layout(template="plotly_dark")
grafico.show()



[*********************100%***********************]  1 of 1 completed


,Close,High,Low,Open,Volume
Date,,,,,
2020-01-02,30.140162,30.201183,29.818399,29.945994,17509700
2020-01-03,29.918259,30.234472,29.724093,29.779568,17284800
2020-01-06,29.740734,29.846138,29.485545,29.846138,32787800
2020-01-07,29.957085,30.062488,29.624229,29.679704,16326400
2020-01-08,29.962633,30.162347,29.746277,30.068039,15298500
...,...,...,...,...,...
2025-12-22,72.919998,73.370003,70.809998,71.029999,27853000
2025-12-23,72.900002,73.529999,72.580002,73.230003,17140900
2025-12-26,73.120003,73.349998,72.620003,72.860001,16368800
